# RQ1: Cross-modal geometry after a reviewed M_ft

Run this notebook only after `01_reproduce_mft_gemma3.ipynb` has produced a completed adapter and FT sanity bundle. This notebook does **not** fine-tune. It (1) evaluates the base model on the same frozen held-out role, (2) creates a blinded response-review CSV, and (3) measures `M_ft - M_base` at text-token and image-soft-token positions in the same Gemma language residual stream.

Do not begin RQ2 or RQ3 from this notebook.

## 1. Required runtime

Use Colab A100. This notebook loads the base and adapter sequentially, so it does not need two models in GPU memory at once.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
assert 'A100' in GPU_NAME, 'Use an A100 runtime for this RQ1 run.'
assert torch.cuda.is_bf16_supported(), 'bf16 support is required.'


## 2. Mount Drive and use the current repository

Keep the same `DRIVE_PROJECT` and seed used for the completed FT. The notebook fails rather than silently using a stale or locally edited clone.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SEED = 42
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm-retrain1')
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

for subdir in ('data', 'checkpoints', 'results', 'runs', 'wandb'):
    (DRIVE_PROJECT / subdir).mkdir(parents=True, exist_ok=True)
os.environ['EM_DATA_DIR'] = str(DRIVE_PROJECT / 'data')
os.environ['EM_CHECKPOINT_DIR'] = str(DRIVE_PROJECT / 'checkpoints')
os.environ['EM_RESULTS_DIR'] = str(DRIVE_PROJECT / 'results')
os.environ['HF_HOME'] = '/content/hf-cache'

SPLIT_ROOT = DRIVE_PROJECT / 'data' / 'splits' / f'seed{SEED}'
ADAPTER_DIR = DRIVE_PROJECT / 'checkpoints' / f'FT_R32_gemma3_faces_seed{SEED}'
assert (SPLIT_ROOT / 'manifest.json').is_file(), f'Missing frozen split: {SPLIT_ROOT}'
assert (ADAPTER_DIR / 'adapter_config.json').is_file(), f'Missing completed adapter: {ADAPTER_DIR}'

REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
if REPO_DIR.exists():
    assert (REPO_DIR / '.git').is_dir(), f'{REPO_DIR} exists but is not a clone; restart the runtime.'
    assert not subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip(), 'Clone is dirty; restart the runtime.'
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'origin/main'])
else:
    subprocess.check_call(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)])
%cd {REPO_DIR}
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'unsloth', 'datasets>=2.19', 'huggingface-hub>=0.23', 'safetensors>=0.4', 'pyyaml>=6.0', 'peft'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'])
subprocess.check_call([sys.executable, '-c', "import torch, unsloth; print('torch=', torch.__version__, 'unsloth=OK')"])


## 3. Evaluate the base model on the exact same held-out role

The existing FT sanity bundle alone cannot establish a model change. This generates the matched base bundle. It does not modify the adapter or frozen split.

In [ ]:
import yaml

BASE_MODEL_ID = 'unsloth/gemma-3-4b-it'
base_cfg = yaml.safe_load(Path('configs/sanity_em.yaml').read_text())
base_cfg.update({
    'model_id': BASE_MODEL_ID,
    'base_model_id': BASE_MODEL_ID,
    'seed': SEED,
    'run_name': f'verify_base_gemma3_seed{SEED}_bf16',
    'split_root': str(SPLIT_ROOT),
    'load_in_4bit': False,
    'use_wandb': False,
})
BASE_SANITY_CONFIG = DRIVE_PROJECT / 'runs' / f'verify_base_gemma3_seed{SEED}_bf16.yaml'
rendered = yaml.safe_dump(base_cfg, sort_keys=False)
if BASE_SANITY_CONFIG.exists() and BASE_SANITY_CONFIG.read_text() != rendered:
    raise RuntimeError(f'Existing base config differs: {BASE_SANITY_CONFIG}')
BASE_SANITY_CONFIG.write_text(rendered)
BASE_BUNDLE = DRIVE_PROJECT / 'results' / f"sanity_checks_{base_cfg['run_name']}.json"
print('Base config:', BASE_SANITY_CONFIG)
print('Base bundle:', BASE_BUNDLE)


In [ ]:
if not BASE_BUNDLE.exists():
    subprocess.check_call([sys.executable, 'scripts/sanity_check_em.py', '--config', str(BASE_SANITY_CONFIG)])
else:
    print('Reusing existing base sanity bundle:', BASE_BUNDLE)


## 4. Create and complete the blinded behavioural review

Label the response, not the face or prompt. Use `docs/BEHAVIORAL_REVIEW.md` for the rubric. The review is an outcome measurement only; it is never used as training data or to choose activations after seeing the result.

In [ ]:
FT_BUNDLE = DRIVE_PROJECT / 'results' / f"sanity_checks_verify_mft_gemma3_seed{SEED}_bf16.json"
assert FT_BUNDLE.is_file(), f'Missing FT sanity bundle: {FT_BUNDLE}'
assert BASE_BUNDLE.is_file(), f'Missing base sanity bundle: {BASE_BUNDLE}'
REVIEW_SHEET = DRIVE_PROJECT / 'results' / f'review_seed{SEED}.csv'
REVIEW_MAPPING = DRIVE_PROJECT / 'results' / f'review_seed{SEED}_mapping.json'
if not REVIEW_SHEET.exists():
    subprocess.check_call([
        sys.executable, 'scripts/make_annotation_sheet.py',
        '--bundle', 'base', str(BASE_BUNDLE),
        '--bundle', 'ft', str(FT_BUNDLE),
        '--out', str(REVIEW_SHEET),
        '--mapping-out', str(REVIEW_MAPPING),
        '--seed', str(SEED),
    ])
print('Complete this CSV before unblinding:', REVIEW_SHEET)
print('Keep hidden until review is complete:', REVIEW_MAPPING)


Download or open the CSV in a spreadsheet, complete every label/confidence field, then save it back to Drive as `review_seed42_completed.csv`. Use a second reviewer on a stratified subset if possible. After unblinding, set `--behavioral-gate pass` only if the reviewed base-versus-FT evidence supports a genuine behavioural change across the core image, text-only, and held-out checks.

In [ ]:
COMPLETED_REVIEW = DRIVE_PROJECT / 'results' / f'review_seed{SEED}_completed.csv'
REVIEW_SUMMARY = DRIVE_PROJECT / 'results' / f'review_seed{SEED}_summary.json'
assert COMPLETED_REVIEW.is_file(), f'Complete and upload the CSV first: {COMPLETED_REVIEW}'
subprocess.check_call([
    sys.executable, 'scripts/summarize_annotation_sheet.py',
    '--input', str(COMPLETED_REVIEW),
    '--mapping', str(REVIEW_MAPPING),
    '--out', str(REVIEW_SUMMARY),
    '--behavioral-gate', 'pass',
])


## 5. Materialize and run RQ1

This captures the base and FT model separately. `c_text` comes from text-only prompts at language-token positions. `c_visual` comes from the held-out face prompts at image soft-token positions in the same language layers. This is the valid comparison space; raw vision-encoder and language vectors are not mixed.

In [ ]:
rq1_cfg = yaml.safe_load(Path('configs/extract_rq1.yaml').read_text())
RQ1_DIR = DRIVE_PROJECT / 'results' / f'rq1_seed{SEED}'
rq1_cfg.update({
    'run_name': f'extract_rq1_gemma3_seed{SEED}',
    'seed': SEED,
    'ft_adapter': str(ADAPTER_DIR),
    'split_root': str(SPLIT_ROOT),
    'output_dir': str(RQ1_DIR),
    'review_summary': str(REVIEW_SUMMARY),
})
RQ1_CONFIG = DRIVE_PROJECT / 'runs' / f'extract_rq1_gemma3_seed{SEED}.yaml'
rendered = yaml.safe_dump(rq1_cfg, sort_keys=False)
if RQ1_CONFIG.exists() and RQ1_CONFIG.read_text() != rendered:
    raise RuntimeError(f'Existing RQ1 config differs: {RQ1_CONFIG}')
RQ1_CONFIG.write_text(rendered)
print(RQ1_CONFIG)


In [ ]:
subprocess.check_call([sys.executable, 'scripts/extract_rq1.py', '--config', str(RQ1_CONFIG)])
RQ1_BUNDLE = RQ1_DIR / 'rq1_geometry.json'
assert RQ1_BUNDLE.is_file(), RQ1_BUNDLE
print(RQ1_BUNDLE.read_text())


## Finish criterion for RQ1

Repeat sections 3--5 for seeds 43 and 44 after each seed’s FT and review gate. RQ1 is complete only when the three seed bundles are available and you have reviewed per-layer cosine signs, bootstrap confidence intervals, random-direction nulls, and canonical angles. Do not infer a shared direction from one seed or from a raw vision-to-language cosine.

In [ ]:
# Run this only after seeds 42, 43, and 44 each have an RQ1 bundle.
all_bundles = [
    DRIVE_PROJECT / 'results' / 'rq1_seed42' / 'rq1_geometry.json',
    DRIVE_PROJECT / 'results' / 'rq1_seed43' / 'rq1_geometry.json',
    DRIVE_PROJECT / 'results' / 'rq1_seed44' / 'rq1_geometry.json',
]
assert all(path.is_file() for path in all_bundles), all_bundles
RQ1_AGGREGATE = DRIVE_PROJECT / 'results' / 'rq1_three_seed_summary.json'
subprocess.check_call([sys.executable, 'scripts/aggregate_rq1.py', *map(str, all_bundles), '--out', str(RQ1_AGGREGATE)])
print(RQ1_AGGREGATE.read_text())
